# Week 12 Live Coding: Reading a Real Regression Table + Synthesis

Two parts. **Part 1 is the capstone of the whole course** — we reproduce a real regression table and read every cell. **Part 2** combines five studies into one honest number.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk12_external_validity_debate/data/gotv_experiment.csv')   # the 2014 GOTV mailer experiment (real data, from W3)
df.head()

## Part 1: Read a real regression table (the capstone)

This is the same `smf.ols` you've run since Week 2 — now with controls. Run it, then we read the table the way you'd read it in a published paper.

**Heads up on the output below:** `.summary()` prints three boxes. Ignore the top box (model bookkeeping) and the bottom box (diagnostics like Durbin-Watson, Omnibus). The **middle box — one row per variable — is the only part you read.** The next cell pulls the six numbers you actually need out of it.

In [ ]:
model = smf.ols('voted_2014 ~ treatment + C(vote_history_stratum) + high_salience_state', data=df).fit()
print(model.summary())

**Reading it, cell by cell:**

1. **Outcome?** `voted_2014` (top of the summary): 1 if the person voted, 0 if not. Every coefficient is a change in the *probability of voting*.
2. **The coefficient you care about** — `treatment`. Everything else is a control.
3. **Standard error and stars** — how precise the estimate is, and whether it clears the significance bar.
4. **Confidence interval** — the `[0.025  0.975]` columns.
5. **Controls** — `C(vote_history_stratum)` and `high_salience_state`; the treatment effect is "holding these fixed."
6. **N and R²** — sample size and how much variation the model explains.

Let's pull the treatment row out explicitly:

In [ ]:
print('Treatment effect:      ', round(100 * model.params['treatment'], 2), 'percentage points')
print('Standard error:        ', round(100 * model.bse['treatment'], 2), 'pp')
print('p-value:               ', round(model.pvalues['treatment'], 4))
ci = model.conf_int().loc['treatment']
print('95% confidence interval:', f'[{100 * ci[0]:+.2f}, {100 * ci[1]:+.2f}]', 'pp')
print('N:', int(model.nobs), '   R-squared:', round(model.rsquared, 3))

So: the mailer raised turnout by **+0.63 points**, p = **0.014** — **one star, significant at 0.05 but not 0.01** — 95% CI **[+0.13, +1.13]**, which excludes zero but barely. A **real, small, imprecise** effect.

*How to judge significance from a table:* let the **stars and the p-value** do it (one star = p < 0.05). As a sanity check, the coefficient (0.0063) is about **2.5x its standard error** — past the rough "2x the SE" bar. Don't try to recompute the exact t from the *rounded* SE the table prints (0.003): rounded numbers give you ~2.1, the unrounded SE (~0.0025) gives 2.5. The lesson for real tables: **trust the reported stars and p; the 2x-SE rule is just an eyeball check.** The control coefficients are far larger (below-median past voters turn out ~48 points less), and that's fine — they aren't what's being tested. N is 107,550, which is why the standard error is so small.

**You just read the main table of a quantitative political-science paper.** That was the goal of the whole course.

## Part 2: Synthesis — five studies, one number

The five canvassing estimates from the case. How do we combine them?

In [ ]:
studies = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk12_external_validity_debate/data/gotv_studies.csv')
print(studies.to_string(index=False))

In [ ]:
# A simple average treats all five equally:
simple = studies['effect_pp'].mean()
print('Simple average:', round(simple, 2), 'pp')

# Precision weighting: weight each study by 1 / SE^2, so sharper estimates count more.
weights = 1 / studies['se_pp']**2
weighted = (weights * studies['effect_pp']).sum() / weights.sum()
print('Precision-weighted average:', round(weighted, 2), 'pp')

The summary falls from **+3.3** to **+1.0 pp**: the big, precise studies (which find small effects) dominate once we stop treating a 600-person study like a 100,000-person one.

In [ ]:
# The publication-bias tell: do SMALL studies report BIGGER effects?
print(studies[['study', 'n', 'effect_pp', 'se_pp']].sort_values('n').to_string(index=False))

Sorted by sample size, the pattern is stark: the **smallest** studies report the **biggest** effects, the largest the smallest. If canvassing had one true effect plus noise, small studies would scatter *symmetrically* around it — instead they're all high. The missing small *null* results are sitting in a file drawer. So even **+1.0** is probably best treated as a soft **ceiling**, not a point estimate.

**Two honest caveats, though.** (1) With only five studies you can't *prove* a file drawer — five points can't run a real publication-bias test. (2) These studies also differ by *setting*: New Haven is a low-salience local race, 2014 is a high-turnout-room midterm at scale. So the same small-study-big-effect pattern could partly be a **genuinely larger effect in low-salience elections** — which you *already* account for when you judge external validity. Don't discount twice: weight by precision, judge transfer, and stay suspicious — but you can't cleanly separate "publication bias" from "real heterogeneity by setting" with five confounded studies.